# Episode 15: The Coordinator Pattern

A `sequence` graph routes the same way every time. But often you want an *agent* to decide where a request goes — that's the **coordinator pattern**.

**In this episode you'll build:** a coordinator that routes a question to the right specialist, then delivers the answer.

This is the Beta replacement for the classic **AutoPattern**.

## Coordinator vs AutoPattern

Classic AG2's `AutoPattern` used an LLM to pick the next speaker — flexible, but the routing logic was hidden inside a speaker-selector.

The coordinator pattern keeps the LLM-driven flexibility but makes the routing **explicit and auditable**:

- A central **coordinator** agent decides where to route.
- It routes by calling a tool that returns a typed **`Handoff`**.
- The `TransitionGraph` sends each specialist's reply back to the coordinator.
- A `finish` tool call terminates the channel.

Every decision is a tool call on the WAL — traceable, testable, and resistant to prompt injection.

## Setup

We need the workflow imports plus `Handoff` (the typed routing return) and the graph conditions/targets.

In [ ]:
from dotenv import load_dotenv

load_dotenv()

import asyncio

from autogen.beta import Agent
from autogen.beta.config import OpenAIConfig
from autogen.beta.knowledge import MemoryKnowledgeStore
from autogen.beta.network import (
    AgentTarget, EV_CHANNEL_CLOSED, EV_PACKET, EV_TEXT, FromSpeaker, Handoff,
    Hub, HubClient, LocalLink, Passport, Resume, TerminateTarget, ToolCalled,
    Transition, TransitionGraph,
)

config = OpenAIConfig(model="gpt-4.1-mini", temperature=0)


async def wait_for_close(hub, channel_id, timeout=240.0):
    deadline = asyncio.get_event_loop().time() + timeout
    while asyncio.get_event_loop().time() < deadline:
        for e in await hub.read_wal(channel_id):
            if e.event_type == EV_CHANNEL_CLOSED:
                return e
        await asyncio.sleep(0.1)
    raise asyncio.TimeoutError("no close")


def turn_text(env):
    if env.event_type == EV_TEXT:
        return env.event_data.get("text")
    if env.event_type == EV_PACKET:
        return env.event_data.get("body")
    return None


## Building the coordinator

The coordinator's routing tools return a `Handoff(target=..., reason=...)` — the framework reads it and routes there. Tools are attached to the `Agent` **before** registration, so the specialists must be registered first (the tools reference their `agent_id`).

In [ ]:
hub = await Hub.open(MemoryKnowledgeStore(), ttl_sweep_interval=0)
link = LocalLink(hub)
u_hc, c_hc, w_hc, s_hc = (HubClient(link, hub=hub) for _ in range(4))

# Register the specialists first -- the coordinator's tools reference them.
weather = await w_hc.register(
    Agent("weather", prompt="You are a weather expert. Answer in one sentence.", config=config),
    Passport(name="weather"), Resume(), attach_plugin=False,
)
sports = await s_hc.register(
    Agent("sports", prompt="You are a sports expert. Answer in one sentence.", config=config),
    Passport(name="sports"), Resume(), attach_plugin=False,
)

# The coordinator routes to ONE specialist, then finishes.
coordinator_agent = Agent("coordinator", prompt=(
    "You route a user question to ONE specialist. "
    "Turn 1: call ask_weather OR ask_sports (whichever fits) exactly once. "
    "Turn 2: after the specialist replies, call finish with their answer "
    "rephrased in one sentence. Never call a routing tool twice."
), config=config)


@coordinator_agent.tool
async def ask_weather(question: str) -> Handoff:
    """Route a weather question to the weather expert."""
    return Handoff(target=weather.agent_id, reason=question)


@coordinator_agent.tool
async def ask_sports(question: str) -> Handoff:
    """Route a sports question to the sports expert."""
    return Handoff(target=sports.agent_id, reason=question)


@coordinator_agent.tool
async def finish(answer: str) -> str:
    """Deliver the final answer to the user."""
    return answer


coordinator = await c_hc.register(
    coordinator_agent, Passport(name="coordinator"), Resume(), attach_plugin=False,
)
user = await u_hc.register_human(Passport(name="user", kind="human"))
print("coordinator + 2 specialists registered")


## The routing graph

A `Handoff` from a tool routes directly to its target. The graph handles the rest: after a specialist speaks, control returns to the coordinator; when the coordinator calls `finish`, the channel terminates.

In [1]:
graph = TransitionGraph(
    initial_speaker=user.agent_id,
    transitions=[
        Transition(when=ToolCalled("finish"), then=TerminateTarget("answered")),
        Transition(when=FromSpeaker(weather.agent_id), then=AgentTarget(coordinator.agent_id)),
        Transition(when=FromSpeaker(sports.agent_id), then=AgentTarget(coordinator.agent_id)),
        Transition(when=FromSpeaker(user.agent_id), then=AgentTarget(coordinator.agent_id)),
    ],
    default_target=TerminateTarget("fall_through"),
    max_turns=8,
)
channel = await user.open(
    type="workflow",
    target=[coordinator.agent_id, weather.agent_id, sports.agent_id],
    knobs={"graph": graph.to_dict()},
)
await channel.send("Is it good running weather to plan a jog this afternoon?")

close_env = await wait_for_close(hub, channel.channel_id)
print(f"closed: {close_env.event_data.get('reason')!r}")

names = {user.agent_id: "user", coordinator.agent_id: "coordinator",
         weather.agent_id: "weather", sports.agent_id: "sports"}
for env in await hub.read_wal(channel.channel_id):
    text = turn_text(env)
    if text and text.strip():
        print(f"[{names[env.sender_id]}] {text}")

for hc in (u_hc, c_hc, w_hc, s_hc):
    await hc.close()
await hub.close()


closed: 'answered'
[user] Is it good running weather to plan a jog this afternoon?
[coordinator] The weather this afternoon is suitable for running, making it a good time to plan your jog.
[weather] The weather this afternoon is suitable for running, making it a good time to plan your jog.
[coordinator] The weather this afternoon is suitable for running, so it is a good time to plan your jog.


## The flow

Trace the close reason `answered` back through the transcript:

1. `user` posts the question — `FromSpeaker(user)` routes to the `coordinator`.
2. The coordinator calls `ask_weather`, which returns a `Handoff` to the `weather` expert.
3. `weather` answers — `FromSpeaker(weather)` routes back to the `coordinator`.
4. The coordinator calls `finish` — `ToolCalled("finish")` fires `TerminateTarget("answered")`.

The coordinator never had hard-coded routing. It *decided*, per request, using its tools — and every decision is on the WAL.

## Additional content

**Terminate rules go first.** The `ToolCalled("finish")` transition is listed before the `FromSpeaker` rules. Order matters — the adapter takes the *first* matching transition, so a terminate rule must be checked before routing rules that could otherwise loop.

**`Handoff` beats the graph.** A typed `Handoff` return routes directly, even overriding a matching `ToolCalled` rule — useful when the *target* depends on runtime state.

**This scales.** Add a third specialist by registering it, adding an `ask_*` tool and a `FromSpeaker` transition. The coordinator pattern is the backbone of triage systems — which is exactly what Episode 17 builds.

## What's next

That completes **Group 3 — Multi-Agent Networking**. You have the Hub and all four adapters. Episode 16 steps back: a single decision framework for choosing between every pattern in the workshop so far.